# 06 — ORESTAR City Council spending profiles (2026)

Candidate-level spending profiles for **Portland City Council** using cleaned
ORESTAR transactions for 2026.

The representation mirrors the fundraising notebooks:

- total spending and expenditure-record count;
- mean, median, minimum, maximum, and standard deviation;
- spending-size bins;
- spending amount share and expenditure-record share by bin.

`expenditure_count` is a count of reported expenditure records, not purchases
or unique vendors.


## 1. Setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 190)

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    ROOT = cwd
elif (cwd.parent / "pyproject.toml").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find repository root. "
        "Expected pyproject.toml in the current directory or its parent."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from helpers.paths import (
    CLEAN,
    PROCESSED,
    fundraising_processed_dir,
    orestar_file_audit_path,
    orestar_transactions_path,
    spending_processed_dir,
)

print("ROOT:", ROOT)

YEAR = 2026
CONTEST = "city_council"

TRANSACTIONS_PATH = orestar_transactions_path(
    YEAR,
    CONTEST,
)

AUDIT_PATH = orestar_file_audit_path(
    YEAR,
    CONTEST,
)

CROSSWALK_PATH = (
    PROCESSED
    / "master"
    / f"candidate_source_crosswalk_{YEAR}.csv"
)

OUTPUT_DIR = spending_processed_dir(
    YEAR,
    CONTEST,
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Transactions:", TRANSACTIONS_PATH)
print("Audit:", AUDIT_PATH)
print("Crosswalk exists:", CROSSWALK_PATH.exists())
print("Output:", OUTPUT_DIR)


ROOT: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis
Transactions: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/clean/orestar/2026/city_council/transactions.csv
Audit: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/clean/orestar/2026/city_council/file_audit.csv
Crosswalk exists: False
Output: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/spending/2026/city_council


## 2. Load and validate cleaned City Council ORESTAR

In [2]:
def to_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )

transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    low_memory=False,
)

required = {
    "year",
    "contest_type",
    "office",
    "district",
    "source_file_stem",
    "sub_type",
    "is_reported_expenditure",
    "reported_expenditure_amount",
}

missing = sorted(
    required
    - set(transactions.columns)
)

if missing:
    raise ValueError(
        f"Missing clean ORESTAR columns: {missing}"
    )

contest_values = set(
    transactions["contest_type"]
    .dropna()
    .astype(str)
    .unique()
)

if contest_values != {CONTEST}:
    raise ValueError(
        f"Expected only {CONTEST}; found {sorted(contest_values)}"
    )

print("Rows:", f"{len(transactions):,}")
display(
    transactions["sub_type"]
    .value_counts(dropna=False)
    .to_frame("rows")
)


Rows: 12,414


,rows
sub_type,
Cash Contribution,8231
Cash Expenditure,3616
Personal Expenditure for Reimbursement,174
In-Kind Contribution,110
Return or Refund of Contribution,107
Items Sold at Fair Market Value,55
Account Payable,41
Loan Received (Non-Exempt),23
Refunds and Rebates,19


## 3. Keep positive reported expenditures

In [3]:
transactions["reported_expenditure_amount"] = pd.to_numeric(
    transactions["reported_expenditure_amount"],
    errors="coerce",
)

spending = transactions.loc[
    to_bool(
        transactions["is_reported_expenditure"]
    )
    & transactions["reported_expenditure_amount"].notna()
    & transactions["reported_expenditure_amount"].gt(0)
].copy()

spending["amount"] = (
    spending["reported_expenditure_amount"]
)

print("Expenditure records:", f"{len(spending):,}")
print("Total spending:", f"${spending['amount'].sum():,.2f}")


Expenditure records: 3,791
Total spending: $2,501,444.38


## 4. File audit

Duplicate source exports are surfaced, not silently removed.


In [4]:
if AUDIT_PATH.exists():
    audit = pd.read_csv(
        AUDIT_PATH,
        low_memory=False,
    )

    duplicate_exports = audit.loc[
        to_bool(
            audit["is_exact_duplicate_export"]
        )
    ].copy()

    print(
        "Exact duplicate export rows:",
        len(duplicate_exports),
    )

    if len(duplicate_exports):
        display(
            duplicate_exports[
                [
                    "district",
                    "source_file",
                    "source_file_hash",
                    "same_hash_file_count",
                ]
            ]
        )
else:
    print("No file audit found.")


Exact duplicate export rows: 0


## 5. Candidate identity

In [5]:
def attach_orestar_identity(data: pd.DataFrame, crosswalk_path: Path) -> pd.DataFrame:
    frame = data.copy()

    frame["source_candidate_name"] = (
        frame["source_file_stem"]
        .astype("string")
        .str.strip()
    )

    frame["canonical_candidate"] = pd.NA
    frame["candidate_key"] = pd.NA
    frame["linkage_status"] = "source_only"

    if crosswalk_path.exists():
        crosswalk = pd.read_csv(
            crosswalk_path,
            low_memory=False,
        )

        needed = {
            "source",
            "classification",
            "year",
            "district",
            "source_candidate_name",
            "suggested_candidate",
            "suggested_candidate_key",
        }

        if needed.issubset(crosswalk.columns):
            matched = (
                crosswalk.loc[
                    crosswalk["source"].eq("orestar")
                    & crosswalk["classification"].eq("match"),
                    [
                        "year",
                        "district",
                        "source_candidate_name",
                        "suggested_candidate",
                        "suggested_candidate_key",
                    ],
                ]
                .rename(
                    columns={
                        "suggested_candidate": "_canonical_candidate",
                        "suggested_candidate_key": "_candidate_key",
                    }
                )
                .drop_duplicates()
            )

            frame = frame.merge(
                matched,
                on=[
                    "year",
                    "district",
                    "source_candidate_name",
                ],
                how="left",
                validate="many_to_one",
            )

            frame["canonical_candidate"] = frame["_canonical_candidate"]
            frame["candidate_key"] = frame["_candidate_key"]

            frame["linkage_status"] = np.where(
                frame["candidate_key"].notna(),
                "matched_to_official_candidate",
                "unmatched_source_candidate",
            )

            frame = frame.drop(
                columns=[
                    "_canonical_candidate",
                    "_candidate_key",
                ]
            )

    frame["candidate"] = (
        frame["canonical_candidate"]
        .fillna(frame["source_candidate_name"])
    )

    source_key = (
        frame["year"].astype("Int64").astype(str)
        + "|"
        + frame["district"].astype("Int64").astype(str)
        + "|orestar|"
        + frame["source_candidate_name"]
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    frame["profile_key"] = (
        frame["candidate_key"]
        .fillna(source_key)
    )

    return frame

spending = attach_orestar_identity(
    spending,
    CROSSWALK_PATH,
)

identity_summary = (
    spending[
        [
            "year",
            "district",
            "source_file_stem",
            "candidate",
            "candidate_key",
            "profile_key",
            "linkage_status",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "district",
            "candidate",
        ]
    )
)

display(identity_summary)

display(
    identity_summary["linkage_status"]
    .value_counts(dropna=False)
    .to_frame("candidate_source_rows")
)


,year,district,source_file_stem,candidate,candidate_key,profile_key,linkage_status
15,2026,3,Corcoran,Corcoran,<NA>,2026|3|orestar|corcoran,source_only
79,2026,3,Johnson,Johnson,<NA>,2026|3|orestar|johnson,source_only
106,2026,3,Koyama,Koyama,<NA>,2026|3|orestar|koyama,source_only
2148,2026,3,Leon,Leon,<NA>,2026|3|orestar|leon,source_only
2245,2026,3,Morillo,Morillo,<NA>,2026|3|orestar|morillo,source_only
3467,2026,3,Mullen,Mullen,<NA>,2026|3|orestar|mullen,source_only
3552,2026,3,Novick,Novick,<NA>,2026|3|orestar|novick,source_only
7143,2026,3,Sollitt,Sollitt,<NA>,2026|3|orestar|sollitt,source_only
7220,2026,3,Zimmerman,Zimmerman,<NA>,2026|3|orestar|zimmerman,source_only
7328,2026,4,Arnold,Arnold,<NA>,2026|4|orestar|arnold,source_only


,candidate_source_rows
linkage_status,
source_only,17


## 6. Candidate spending summary

In [6]:
PROFILE_KEYS = [
    "year",
    "contest_type",
    "office",
    "district",
    "profile_key",
    "candidate",
]

candidate_summary = (
    spending
    .groupby(
        PROFILE_KEYS,
        as_index=False,
        dropna=False,
    )
    .agg(
        total_spending=("amount", "sum"),
        expenditure_count=("amount", "size"),
        average_expenditure=("amount", "mean"),
        median_expenditure=("amount", "median"),
        min_expenditure=("amount", "min"),
        max_expenditure=("amount", "max"),
        std_expenditure=("amount", "std"),
    )
)

identity_cols = (
    spending[
        [
            "profile_key",
            "source_file_stem",
            "source_candidate_name",
            "canonical_candidate",
            "candidate_key",
            "linkage_status",
        ]
    ]
    .drop_duplicates(
        subset=["profile_key"]
    )
)

candidate_summary = candidate_summary.merge(
    identity_cols,
    on="profile_key",
    how="left",
    validate="one_to_one",
)

candidate_summary = candidate_summary.sort_values(
    [
        "district",
        "total_spending",
    ],
    ascending=[
        True,
        False,
    ],
)

display(candidate_summary)


,year,contest_type,office,district,profile_key,candidate,total_spending,expenditure_count,average_expenditure,median_expenditure,min_expenditure,max_expenditure,std_expenditure,source_file_stem,source_candidate_name,canonical_candidate,candidate_key,linkage_status
6,2026,city_council,Portland City Council,3,2026|3|orestar|novick,Novick,1081644.06,662,1633.903414,129.525,0.35,75039.00,5813.440766,Novick,Novick,<NA>,<NA>,source_only
2,2026,city_council,Portland City Council,3,2026|3|orestar|koyama,Koyama,311196.92,912,341.224693,21.530,0.04,29165.00,1818.961262,Koyama,Koyama,<NA>,<NA>,source_only
4,2026,city_council,Portland City Council,3,2026|3|orestar|morillo,Morillo,237544.36,522,455.065824,30.000,0.20,20000.00,1763.415537,Morillo,Morillo,<NA>,<NA>,source_only
8,2026,city_council,Portland City Council,3,2026|3|orestar|zimmerman,Zimmerman,7901.03,35,225.743714,29.000,0.87,1500.00,355.171968,Zimmerman,Zimmerman,<NA>,<NA>,source_only
0,2026,city_council,Portland City Council,3,2026|3|orestar|corcoran,Corcoran,2879.67,33,87.262727,29.480,0.20,780.00,148.108215,Corcoran,Corcoran,<NA>,<NA>,source_only
7,2026,city_council,Portland City Council,3,2026|3|orestar|sollitt,Sollitt,2660.56,40,66.514000,25.000,0.20,540.00,109.473492,Sollitt,Sollitt,<NA>,<NA>,source_only
3,2026,city_council,Portland City Council,3,2026|3|orestar|leon,Leon,1986.64,33,60.201212,18.610,0.20,505.50,99.016001,Leon,Leon,<NA>,<NA>,source_only
5,2026,city_council,Portland City Council,3,2026|3|orestar|mullen,Mullen,1075.23,8,134.403750,4.915,3.50,1000.00,349.972752,Mullen,Mullen,<NA>,<NA>,source_only
1,2026,city_council,Portland City Council,3,2026|3|orestar|johnson,Johnson,389.46,13,29.958462,13.190,0.17,115.00,34.258506,Johnson,Johnson,<NA>,<NA>,source_only
16,2026,city_council,Portland City Council,4,2026|4|orestar|zimmerman,Zimmerman,293433.34,192,1528.298646,156.640,0.35,50000.00,5207.707704,Zimmerman,Zimmerman,<NA>,<NA>,source_only


## 7. Expenditure-size bins

We reuse the same numeric boundaries as the fundraising profiles only as a
simple scale representation. These are not substantive spending-purpose
categories.


In [7]:
BIN_LABELS = [
    "Micro",
    "Small",
    "Medium",
    "Large",
    "Mega",
]

BIN_EDGES = [
    -np.inf,
    25,
    100,
    250,
    1000,
    np.inf,
]

spending["spending_bin"] = pd.cut(
    spending["amount"],
    bins=BIN_EDGES,
    labels=BIN_LABELS,
    right=True,
    ordered=True,
)

display(
    spending["spending_bin"]
    .value_counts(sort=False)
    .to_frame("records")
)


,records
spending_bin,
Micro,1608
Small,897
Medium,401
Large,465
Mega,420


## 8. Long spending profile

In [8]:
profile_long = (
    spending
    .groupby(
        PROFILE_KEYS
        + [
            "spending_bin",
        ],
        observed=False,
        as_index=False,
        dropna=False,
    )
    .agg(
        spending_amount=("amount", "sum"),
        spending_count=("amount", "size"),
    )
)

profile_totals = (
    profile_long
    .groupby(
        PROFILE_KEYS,
        as_index=False,
        dropna=False,
    )
    .agg(
        profile_total_spending=("spending_amount", "sum"),
        profile_total_count=("spending_count", "sum"),
    )
)

profile_long = profile_long.merge(
    profile_totals,
    on=PROFILE_KEYS,
    how="left",
    validate="many_to_one",
)

profile_long["spending_amount_share"] = (
    profile_long["spending_amount"]
    / profile_long["profile_total_spending"]
)

profile_long["spending_count_share"] = (
    profile_long["spending_count"]
    / profile_long["profile_total_count"]
)

profile_long = profile_long.merge(
    identity_cols,
    on="profile_key",
    how="left",
    validate="many_to_one",
)

display(profile_long.head(15))


,year,contest_type,office,district,profile_key,candidate,spending_bin,spending_amount,spending_count,profile_total_spending,profile_total_count,spending_amount_share,spending_count_share,source_file_stem,source_candidate_name,canonical_candidate,candidate_key,linkage_status
0,2026,city_council,Portland City Council,3,2026|3|orestar|corcoran,Corcoran,Micro,127.68,16,2879.67,33,0.044338,0.484848,Corcoran,Corcoran,<NA>,<NA>,source_only
1,2026,city_council,Portland City Council,3,2026|3|orestar|corcoran,Corcoran,Small,545.70,8,2879.67,33,0.189501,0.242424,Corcoran,Corcoran,<NA>,<NA>,source_only
2,2026,city_council,Portland City Council,3,2026|3|orestar|corcoran,Corcoran,Medium,1111.31,7,2879.67,33,0.385916,0.212121,Corcoran,Corcoran,<NA>,<NA>,source_only
3,2026,city_council,Portland City Council,3,2026|3|orestar|corcoran,Corcoran,Large,1094.98,2,2879.67,33,0.380245,0.060606,Corcoran,Corcoran,<NA>,<NA>,source_only
4,2026,city_council,Portland City Council,3,2026|3|orestar|corcoran,Corcoran,Mega,0.00,0,2879.67,33,0.000000,0.000000,Corcoran,Corcoran,<NA>,<NA>,source_only
5,2026,city_council,Portland City Council,3,2026|3|orestar|johnson,Johnson,Micro,33.30,7,389.46,13,0.085503,0.538462,Johnson,Johnson,<NA>,<NA>,source_only
6,2026,city_council,Portland City Council,3,2026|3|orestar|johnson,Johnson,Small,241.16,5,389.46,13,0.619216,0.384615,Johnson,Johnson,<NA>,<NA>,source_only
7,2026,city_council,Portland City Council,3,2026|3|orestar|johnson,Johnson,Medium,115.00,1,389.46,13,0.295281,0.076923,Johnson,Johnson,<NA>,<NA>,source_only
8,2026,city_council,Portland City Council,3,2026|3|orestar|johnson,Johnson,Large,0.00,0,389.46,13,0.000000,0.000000,Johnson,Johnson,<NA>,<NA>,source_only
9,2026,city_council,Portland City Council,3,2026|3|orestar|johnson,Johnson,Mega,0.00,0,389.46,13,0.000000,0.000000,Johnson,Johnson,<NA>,<NA>,source_only


## 9. Validate profile shares

In [9]:
validation = (
    profile_long
    .groupby(
        PROFILE_KEYS,
        as_index=False,
        dropna=False,
    )
    .agg(
        amount_share_sum=("spending_amount_share", "sum"),
        count_share_sum=("spending_count_share", "sum"),
    )
)

validation["amount_share_ok"] = np.isclose(
    validation["amount_share_sum"],
    1.0,
)

validation["count_share_ok"] = np.isclose(
    validation["count_share_sum"],
    1.0,
)

print(
    "All amount-share profiles valid:",
    validation["amount_share_ok"].all(),
)

print(
    "All count-share profiles valid:",
    validation["count_share_ok"].all(),
)

display(
    validation.loc[
        ~validation["amount_share_ok"]
        | ~validation["count_share_ok"]
    ]
)


All amount-share profiles valid: True
All count-share profiles valid: True


,year,contest_type,office,district,profile_key,candidate,amount_share_sum,count_share_sum,amount_share_ok,count_share_ok


## 10. Wide spending profile

In [10]:
metrics = [
    "spending_amount",
    "spending_amount_share",
    "spending_count",
    "spending_count_share",
]

wide_parts = []

for metric in metrics:
    part = (
        profile_long
        .pivot(
            index=PROFILE_KEYS,
            columns="spending_bin",
            values=metric,
        )
        .reindex(columns=BIN_LABELS)
        .fillna(0)
    )

    part.columns = [
        f"{metric}_{str(bin_name).lower()}"
        for bin_name in part.columns
    ]

    wide_parts.append(part)

profile_wide = pd.concat(
    wide_parts,
    axis=1,
).reset_index()

profile_wide = profile_wide.merge(
    candidate_summary,
    on=PROFILE_KEYS,
    how="left",
    validate="one_to_one",
)

print("Rows:", len(profile_wide))
print("Columns:", len(profile_wide.columns))
display(profile_wide.head())


Rows: 17
Columns: 38


,year,contest_type,office,district,profile_key,candidate,spending_amount_micro,spending_amount_small,spending_amount_medium,spending_amount_large,spending_amount_mega,spending_amount_share_micro,spending_amount_share_small,spending_amount_share_medium,spending_amount_share_large,spending_amount_share_mega,spending_count_micro,spending_count_small,spending_count_medium,spending_count_large,spending_count_mega,spending_count_share_micro,spending_count_share_small,spending_count_share_medium,spending_count_share_large,spending_count_share_mega,total_spending,expenditure_count,average_expenditure,median_expenditure,min_expenditure,max_expenditure,std_expenditure,source_file_stem,source_candidate_name,canonical_candidate,candidate_key,linkage_status
0,2026,city_council,Portland City Council,3,2026|3|orestar|corcoran,Corcoran,127.68,545.70,1111.31,1094.98,0.00,0.044338,0.189501,0.385916,0.380245,0.000000,16,8,7,2,0,0.484848,0.242424,0.212121,0.060606,0.000000,2879.67,33,87.262727,29.48,0.20,780.0,148.108215,Corcoran,Corcoran,<NA>,<NA>,source_only
1,2026,city_council,Portland City Council,3,2026|3|orestar|johnson,Johnson,33.30,241.16,115.00,0.00,0.00,0.085503,0.619216,0.295281,0.000000,0.000000,7,5,1,0,0,0.538462,0.384615,0.076923,0.000000,0.000000,389.46,13,29.958462,13.19,0.17,115.0,34.258506,Johnson,Johnson,<NA>,<NA>,source_only
2,2026,city_council,Portland City Council,3,2026|3|orestar|koyama,Koyama,3863.10,13297.90,11021.11,26204.45,256810.36,0.012414,0.042731,0.035415,0.084205,0.825234,480,249,62,51,70,0.526316,0.273026,0.067982,0.055921,0.076754,311196.92,912,341.224693,21.53,0.04,29165.0,1818.961262,Koyama,Koyama,<NA>,<NA>,source_only
3,2026,city_council,Portland City Council,3,2026|3|orestar|leon,Leon,197.84,465.29,818.01,505.50,0.00,0.099585,0.234210,0.411756,0.254450,0.000000,19,8,5,1,0,0.575758,0.242424,0.151515,0.030303,0.000000,1986.64,33,60.201212,18.61,0.20,505.5,99.016001,Leon,Leon,<NA>,<NA>,source_only
4,2026,city_council,Portland City Council,3,2026|3|orestar|morillo,Morillo,1685.41,6468.99,7897.77,33612.72,187879.47,0.007095,0.027233,0.033248,0.141501,0.790924,217,141,53,61,50,0.415709,0.270115,0.101533,0.116858,0.095785,237544.36,522,455.065824,30.00,0.20,20000.0,1763.415537,Morillo,Morillo,<NA>,<NA>,source_only


## 11. Quick descriptive view

In [11]:
district_summary = (
    candidate_summary
    .groupby(
        [
            "year",
            "district",
        ],
        as_index=False,
    )
    .agg(
        candidate_source_rows=("profile_key", "nunique"),
        total_spending=("total_spending", "sum"),
        median_candidate_spending=("total_spending", "median"),
        min_candidate_spending=("total_spending", "min"),
        max_candidate_spending=("total_spending", "max"),
    )
)

display(district_summary)


,year,district,candidate_source_rows,total_spending,median_candidate_spending,min_candidate_spending,max_candidate_spending
0,2026,3,9,1647277.93,2879.670,389.46,1081644.06
1,2026,4,8,854166.45,73328.255,330.89,293433.34


## 12. Export

In [12]:
summary_path = (
    OUTPUT_DIR
    / "orestar_candidate_spending_summary.csv"
)

long_path = (
    OUTPUT_DIR
    / "orestar_candidate_spending_profiles_long.csv"
)

wide_path = (
    OUTPUT_DIR
    / "orestar_candidate_spending_profiles_wide.csv"
)

candidate_summary.to_csv(
    summary_path,
    index=False,
)

profile_long.to_csv(
    long_path,
    index=False,
)

profile_wide.to_csv(
    wide_path,
    index=False,
)

print("SAVED", summary_path)
print("SAVED", long_path)
print("SAVED", wide_path)


SAVED /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/spending/2026/city_council/orestar_candidate_spending_summary.csv
SAVED /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/spending/2026/city_council/orestar_candidate_spending_profiles_long.csv
SAVED /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/spending/2026/city_council/orestar_candidate_spending_profiles_wide.csv
